## 0. Imports

In [22]:
pip install matplotlib plotly gensim scikit-learn bertopic spacy pip pyLDAvis

/opt/python/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=3829) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import textwrap
import urllib.request
import pyLDAvis
import pyLDAvis.lda_model


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import plotly.express as px
import networkx as nx
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.feature_extraction.text import CountVectorizer

from src.data_loader import load_metadata, load_texts
from src.preprocessing import lemmatize, CUSTOM_STOPWORDS, TOKEN_PATTERN
from src.lda import build_lda, plot_top_words
from src.nmf import build_nmf
from src.bertopic_train import build_topic_model, fit_topic_model
from src.party_analysis import build_party_profiles, pca_coords, build_similarity_graph, build_analysis_df

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="multiprocessing")


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Chargement des données

In [2]:
df = load_metadata("data/archelect_search.csv")
df = load_texts(df)

In [ ]:
print(df["text"].iloc[0])

## 2. Exploration du corpus

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
year_counts = df["year"].value_counts().sort_index()
ax.bar(year_counts.index.astype(str), year_counts.values, color="steelblue", edgecolor="white")
ax.set_title("Professions de foi par année d'élection")
ax.set_xlabel("Année"); ax.set_ylabel("Nombre de documents")
for bar, val in zip(ax.patches, year_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(val), ha="center", va="bottom", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
TOP_N = 25
party_counts = (
    df["parti"].replace("non mentionné", None).dropna()
    .value_counts().head(TOP_N)
)
fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(party_counts.index[::-1], party_counts.values[::-1], color="steelblue", edgecolor="white")
ax.set_title(f"Top {TOP_N} des partis/listes")
ax.set_xlabel("Nombre de candidats")
for bar, val in zip(bars, party_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, str(val), va="center", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sexe_counts = df["titulaire-sexe"].replace("non mentionné", None).dropna().value_counts()
axes[0].pie(sexe_counts.values, labels=sexe_counts.index, autopct="%1.1f%%",
            colors=["steelblue", "coral", "brown"], startangle=90)
axes[0].set_title("Répartition par sexe")

age_order = ["moins de 30 ans", "entre 30 et 39 ans", "entre 40 et 49 ans",
             "entre 50 et 59 ans", "entre 60 et 69 ans", "70 ans et plus"]
age_counts = (
    df["titulaire-age-tranche"].replace("non mentionné", None).dropna()
    .value_counts().reindex(age_order).dropna()
)
axes[1].bar(range(len(age_counts)), age_counts.values, color="mediumseagreen", edgecolor="white")
axes[1].set_xticks(range(len(age_counts)))
axes[1].set_xticklabels(age_counts.index, rotation=30, ha="right")
axes[1].set_title("Répartition par tranche d'âge")
plt.tight_layout(); plt.show()

In [ ]:
df_sexe = df[df["titulaire-sexe"].isin(["homme", "femme"])]
femme_pct = df_sexe.groupby("year")["titulaire-sexe"].apply(lambda s: (s == "femme").mean() * 100)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(femme_pct.index, femme_pct.values, marker="o", color="coral", linewidth=2)
ax.set_title("Part des candidatures féminines par élection (%)")
ax.set_xlabel("Année"); ax.set_ylabel("% de femmes")
ax.set_xticks(femme_pct.index)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}"))
for x, y in zip(femme_pct.index, femme_pct.values):
    ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout(); plt.show()

In [ ]:
prof_counts = (
    df["titulaire-profession"].replace("non mentionné", None).dropna()
    .str.split(";").str[0].str.strip().value_counts().head(20)
)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(prof_counts.index[::-1], prof_counts.values[::-1], color="mediumseagreen", edgecolor="white")
ax.set_title("Top 20 des professions des titulaires")
ax.set_xlabel("Nombre de candidats")
plt.tight_layout(); plt.show()

## 3. Prétraitement — Lemmatisation

In [3]:
df = lemmatize(df)

21167 textes déjà lemmatisés, 0 restants


## 4. LDA

In [ ]:
tf_vectorizer, tf, lda = build_lda(df, n_topics=10, n_features=1000)

In [ ]:
plot_top_words(lda, tf_vectorizer, 10, "LDA Topics — ARCHELEC", save_path="figure/figure_6.png")


In [ ]:
pyLDAvis.enable_notebook()
pyLDAvis.lda_model.prepare(lda, tf, tf_vectorizer)

In [ ]:
vis = pyLDAvis.lda_model.prepare(lda, tf, tf_vectorizer)
pyLDAvis.save_html(vis, "plot_interactif/lda_visualization.html")

In [ ]:
# --- Topics par document ---
lda_doc_topic = lda.transform(tf)  # (n_docs, n_topics)
lda_dominant = np.argmax(lda_doc_topic, axis=1)

lda_df = pd.DataFrame({"id": df["id"].values, "lda_topic": lda_dominant})
lda_df.to_csv("data/lda/lda_topics.csv", index=False)


# --- Mots par topic ---
feature_names_lda = tf_vectorizer.get_feature_names_out()
topic_words = [
    {
        "topic": i,
        "top_words": ", ".join(feature_names_lda[j] for j in topic.argsort()[:-16:-1]),
    }
    for i, topic in enumerate(lda.components_)
]
pd.DataFrame(topic_words).to_csv("data/lda/lda_topic_words.csv", index=False)


## 5. NMF

In [ ]:
tfidf_vectorizer, tfidf, nmf, W_normalized, df = build_nmf(df, n_topics=10)


feature_names = tfidf_vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in np.argsort(-topic)[:15]]
    print(f"Topic {topic_idx:2d} ")
    print(f"  {', '.join(top_words)}\n")

In [ ]:
plot_top_words(nmf, tfidf_vectorizer, 15, "NMF Topics — ARCHELEC", save_path="figure/figure_7.png")


In [ ]:
# --- Topics par document ---
nmf_doc_topic = W_normalized  # déjà calculé par build_nmf (n_docs, n_topics)
nmf_dominant = np.argmax(nmf_doc_topic, axis=1)

nmf_df = pd.DataFrame({"id": df["id"].values, "nmf_topic": nmf_dominant})
nmf_df.to_csv("data/nmf/nmf_topics.csv", index=False)


# --- Mots par topic ---
feature_names_nmf = tfidf_vectorizer.get_feature_names_out()
topic_words = [
    {
        "topic": i,
        "top_words": ", ".join(feature_names_nmf[j] for j in topic.argsort()[:-16:-1]),
    }
    for i, topic in enumerate(nmf.components_)
]
pd.DataFrame(topic_words).to_csv("data/nmf/nmf_topic_words.csv", index=False)


In [ ]:
topic_label = {
    0: "Socialisme et autogestion ouvrière",
    1: "Front populaire et justice fiscale",
    2: "Lutte ouvrière et féminisme",
    3: "Écologie animaliste et protection nature",
    4: "Immigration et sécurité nationale",
    5: "Écologie politique et Les Verts",
    6: "Institutions républicaines et élections",
    7: "Anticapitalisme et critique du patronat",
    8: "Social-démocratie et développement",
    9: "Réforme sociale et pouvoir d'achat",
}

In [ ]:
topic_names = [f"{k} — {topic_label[k]}" for k in topic_label]

for _, row in df.sample(n=5, random_state=2026).iterrows():
    idx = row.name
    dominant = row["dominant_topic"]
    print(f"\nDocument {idx} | {row['titulaire-nom']} {row['titulaire-prenom']} | {row['parti']} | {row['year']}")
    print(f"Topic dominant : {dominant} — {topic_label[dominant]} ({row['topic_score']:.3f})")
    print(textwrap.fill(row["text"], width=100)[:600])
    plt.figure(figsize=(14, 3))
    bars = plt.bar(range(len(topic_names)), W_normalized[idx])
    bars[dominant].set_color("darkred")
    plt.xticks(range(len(topic_names)), topic_names, rotation=45, ha="right", fontsize=8)
    plt.ylabel("Proportion")
    plt.title(f"{row['titulaire-nom']} {row['titulaire-prenom']} ({row['year']})")
    plt.tight_layout(); plt.show()

In [ ]:
from sklearn.decomposition import NMF as _NMF

analyzer = tfidf_vectorizer.build_analyzer()
tokenized_texts = [analyzer(doc) for doc in df["lemmatized_text"].fillna("")]
dictionary = Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=10, no_above=0.95)

def _get_topics(model, fnames, n=10):
    return [[fnames[i] for i in np.argsort(-t)[:n]] for t in model.components_]

topic_range = [5, 10, 15]
coherence_metrics = ["u_mass", "c_v", "c_uci", "c_npmi"]
results = {m: {} for m in coherence_metrics}

for k in topic_range:
    nmf_k = _NMF(n_components=k, random_state=42)
    nmf_k.fit(tfidf)
    topics_k = _get_topics(nmf_k, feature_names)
    for m in coherence_metrics:
        results[m][k] = CoherenceModel(
            topics=topics_k, texts=tokenized_texts,
            dictionary=dictionary, coherence=m
        ).get_coherence()
        print(f"k={k:2d} | {m:8s} : {results[m][k]:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
for m in coherence_metrics:
    ks = sorted(results[m])
    scores = np.array([results[m][k] for k in ks])
    norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    plt.plot(ks, norm, marker="o", label=m)
plt.xlabel("Nombre de topics (k)"); plt.ylabel("Cohérence normalisée")
plt.title("Choix du nombre de topics — NMF ARCHELEC")
plt.xticks(topic_range); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

## 6. BERTopic

In [4]:
docs = df["lemmatized_text"].fillna("").tolist()

EMBEDDINGS_PATH = "data/embeddings/embeddings.npy"
if os.path.exists(EMBEDDINGS_PATH):
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"Embeddings chargés : {embeddings.shape}")
else:
    from sentence_transformers import SentenceTransformer
    model_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    embeddings = model_emb.encode(docs, show_progress_bar=True, batch_size=64)
    os.makedirs("data/embeddings", exist_ok=True)
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Embeddings calculés et sauvegardés : {embeddings.shape}")

Embeddings chargés : (21167, 384)


In [5]:
# Filtrage vocabulaire sur les docs individuels (comme LDA et NMF)
vocab_cv = CountVectorizer(
    stop_words=CUSTOM_STOPWORDS,
    token_pattern=TOKEN_PATTERN,
    min_df=5,
    max_df=0.95,
)
vocab_cv.fit(docs)

topic_model = build_topic_model(
    CUSTOM_STOPWORDS,
    TOKEN_PATTERN,
    nr_topics=10,
    umap_n_neighbors=10,
    umap_n_components=10,
    hdbscan_min_cluster_size=10,
    vocabulary=vocab_cv.vocabulary_,   
)
topics, probs = fit_topic_model(topic_model, docs, embeddings)
df["bertopic_topic"] = topics
topic_model.get_topic_info()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4786.70it/s]
2026-04-30 22:25:47,207 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-30 22:26:48,110 - BERTopic - Dimensionality - Completed ✓
2026-04-30 22:26:48,118 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-30 22:26:51,384 - BERTopic - Cluster - Completed ✓
2026-04-30 22:26:51,389 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-30 22:26:59,300 - BERTopic - Representation - Completed ✓
2026-04-30 22:26:59,303 - BERTopic - Topic reduction - Reducing number of topics
2026-04-30 22:26:59,326 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-30 22:27:07,318 - BERTopic - Representation - Completed ✓
2026-04-30 22:27:07,328 - BERTopic - Topic reduction - Reduced number of topics from 217 to 10


,Topic,Count,Name,Representation,Representative_Docs
0,-1,12284,-1_social_majorite_devoir_vie,"[social, majorite, devoir, vie, emploi, maire,...",[\n election legislatif de 14 juin 1981 \n pou...
1,0,6710,0_travailleur_social_devoir_femme,"[travailleur, social, devoir, femme, majorite,...",[election legislatives de mars 1978 5 degre ci...
2,1,1270,1_ecologie_nature_animaux_entente,"[ecologie, nature, animaux, entente, generatio...",[\n REPUBLIQUE francais - election legislatif ...
3,2,399,2_travailleur_payer_patron_maintenir,"[travailleur, payer, patron, maintenir, patron...",[election legislatif de 21 mars 1993 \n pourqu...
4,3,297,3_travailleur_socialisme_ouvrier_capitalisme,"[travailleur, socialisme, ouvrier, capitalisme...",[ligue communiste section francais de le quatr...
5,4,91,4_accepter_triage_europe_illusion,"[accepter, triage, europe, illusion, grand, so...","[parti ouvrier europeen Siege : 19 , rue Nolle..."
6,5,44,5_emploi_mois_retraite_familial,"[emploi, mois, retraite, familial, pourcent, v...",[\n election legislatif 21 mars 1993 de le 4 e...
7,6,43,6_ecologie_entente_generation_vie,"[ecologie, entente, generation, vie, environne...",[\n entente de ecologiste \n pour votre enviro...
8,7,15,7_loi_criminalite_coherence_reduction,"[loi, criminalite, coherence, reduction, group...",[election legislatives mars 1993 \n Parti de l...
9,8,14,8_oder_gesellschaft_ressource_um,"[oder, gesellschaft, ressource, um, natur, was...",[ecologie 78 \n ecologie et SURVIE \n monsieur...


In [6]:
tokenized = [doc.split() for doc in docs]
dictionary_bert = Dictionary(tokenized)

topics_words = [
    [w for w, _ in topic_model.get_topic(t)][:10]
    for t in topic_model.get_topics() if t != -1
]
cm_umass = CoherenceModel(topics=topics_words, texts=tokenized,
                          dictionary=dictionary_bert, coherence="u_mass")
cm_cv    = CoherenceModel(topics=topics_words, texts=tokenized,
                          dictionary=dictionary_bert, coherence="c_v")
print(f"u_mass : {cm_umass.get_coherence():.4f}")
print(f"c_v    : {cm_cv.get_coherence():.4f}")

u_mass : -1.3728
c_v    : 0.5855


In [7]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings)

df["topic"] = topics
df["topic_reduc"] = new_topics


In [ ]:
fig = topic_model.visualize_documents(
    docs=docs, embeddings=embeddings,
    hide_annotations=True, topics=list(range(15)),
    height=600, width=1000
)
fig.write_html("plot_interactif/visu_documents.html")
fig


In [15]:
fig = topic_model.visualize_barchart(n_words=20, topics=list(range(9)))
fig.write_html("plot_interactif/bertopic_barchart.html")
fig


In [11]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame([
    {"topic": topic_id, "mots": ", ".join(w for w, _ in words)}
    for topic_id, words in topic_model.topic_representations_.items()
])


,topic,mots
0,-1,"social, majorite, devoir, vie, emploi, maire, vouloir, pays, liberte, falloir"
1,0,"travailleur, social, devoir, femme, majorite, gouvernement, vouloir, falloir, pays, homme"
2,1,"ecologie, nature, animaux, entente, generation, humain, environnement, vie, homme, verts"
3,2,"travailleur, payer, patron, maintenir, patronat, devoir, revolutionnaire, sacrifice, jeter, vendre"
4,3,"travailleur, socialisme, ouvrier, capitalisme, bourgeoisie, capitaliste, vouloir, falloir, societe, revendication"
5,4,"accepter, triage, europe, illusion, grand, sortir, financier, technologie, production, detruire"
6,5,"emploi, mois, retraite, familial, pourcent, vie, logement, changement, vivre, social"
7,6,"ecologie, entente, generation, vie, environnement, verts, humain, chance, donner, nature"
8,7,"loi, criminalite, coherence, reduction, groupe, connaissance, univers, graine, nature, conscience"
9,8,"oder, gesellschaft, ressource, um, natur, was, viel, verschwendung, unsere, brauchen"


In [12]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.write_html("plot_interactif/bertopic_hierarchy.html")
fig

100%|██████████| 8/8 [00:00<00:00, 65.59it/s]


In [13]:
fig = topic_model.visualize_heatmap()
fig.write_html("plot_interactif/bertopic_heatmap.html")
fig


In [14]:
topic_distr_sample, _ = topic_model.approximate_distribution(docs[:1], window=4, stride=1)
fig = topic_model.visualize_distribution(topic_distr_sample[0])
fig.write_html("plot_interactif/bertopic_distribution.html")
fig


100%|██████████| 1/1 [00:00<00:00, 190.74it/s]


In [16]:
topic_labels = {
    -1: "Outliers",
    0: "Travailleurs, femmme et gouvernement",
    1: "Écologie, environnement et entente",
    2: "Capital VS travail",
    3: "Socialisme et capitalisme",
    4: "Europe, illusion et économie",
    5: "Emploi, retraite, vie familiale et logement",
    6: "Écologie et génération",
    7: "Loi etcriminalité",
    8: "Gesellschaft und Ressourcen (Alsace)",

}



df["topic_label"] = df["topic"].map(topic_labels)
topic_model.set_topic_labels(topic_labels)



# Sauvegarde
os.makedirs("data/BERTopic", exist_ok=True)
topic_model.save("data/BERTopic/bertopic_model")
df[["id", "topic", "topic_label"]].to_csv("data/BERTopic/df_topics.csv", index=False)
print("Modèle et topics sauvegardés.")

2026-04-30 22:53:04,651 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Modèle et topics sauvegardés.


## 7. Carte sémantique des partis

In [ ]:
DISTR_PATH = "data/BERTopic/topic_distr.npy"
if os.path.exists(DISTR_PATH):
    topic_distr = np.load(DISTR_PATH)
else:
    topic_distr, _ = topic_model.approximate_distribution(docs, window=4, stride=1)
    np.save(DISTR_PATH, topic_distr)

distr_df = pd.DataFrame(
    topic_distr,
    columns=[topic_labels[i] for i in range(len(topic_labels) - 1)]
)
distr_df["parti"] = df["parti"].values

party_profiles, dominant_topic, topic_cols = build_party_profiles(distr_df)
coords, pca, colors, color_map, unique_topics = pca_coords(party_profiles, dominant_topic)
print(f"{len(party_profiles)} partis | {len(topic_cols)} topics")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=100, alpha=0.8)
for i, parti in enumerate(party_profiles.index):
    ax.annotate(parti, (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.85)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Carte sémantique des partis politiques (1973–1993)")
legend = [Patch(color=color_map[t], label=t) for t in unique_topics]
ax.legend(handles=legend, fontsize=7, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig("data/carte_semantique_partis.png", dpi=150)
plt.show()

In [ ]:
n_docs_by_parti = distr_df.groupby("parti").size()
fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=100, alpha=0.8)
for i, parti in enumerate(party_profiles.index):
    if n_docs_by_parti[parti] >= n_docs_by_parti.nlargest(30).min():
        ax.annotate(parti, (coords[i, 0], coords[i, 1]), fontsize=8)
plt.title("Carte sémantique — Top 30 partis annotés")
plt.tight_layout(); plt.show()

In [ ]:
coords3, pca3, *_ = pca_coords(party_profiles, dominant_topic, n_components=3)
plot_df3 = pd.DataFrame({
    "x": coords3[:, 0], "y": coords3[:, 1], "z": coords3[:, 2],
    "parti": party_profiles.index,
    "topic": dominant_topic.values,
    "n_docs": distr_df.groupby("parti").size().values,
})
fig = px.scatter_3d(
    plot_df3, x="x", y="y", z="z",
    color="topic", hover_name="parti", size="n_docs",
    title="Carte sémantique 3D des partis (1973–1993)",
    labels={
        "x": f"PC1 ({pca3.explained_variance_ratio_[0]:.1%})",
        "y": f"PC2 ({pca3.explained_variance_ratio_[1]:.1%})",
        "z": f"PC3 ({pca3.explained_variance_ratio_[2]:.1%})",
    }
)
fig.show()

In [ ]:
G = build_similarity_graph(party_profiles, dominant_topic, distr_df, color_map, seuil=0.8)

fig, ax = plt.subplots(figsize=(18, 14))
pos = nx.spring_layout(G, k=2, seed=42)
node_colors = [color_map.get(G.nodes[n]["topic"], "grey") for n in G.nodes]
node_sizes  = [max(G.nodes[n]["size"] * 2, 50) for n in G.nodes]
edge_weights = [G[u][v]["weight"] for u, v in G.edges]
nx.draw_networkx_edges(G, pos, alpha=0.3, width=[w * 2 for w in edge_weights], ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title("Réseau de similarité thématique entre partis (1973–1993)")
ax.axis("off")
legend = [Patch(color=color_map[t], label=t) for t in unique_topics]
ax.legend(handles=legend, fontsize=7, loc="upper left")
plt.tight_layout()
plt.savefig("data/reseau_partis.png", dpi=150)
plt.show()

## 8. Analyse par métadonnées

In [ ]:
analysis_df, topic_cols = build_analysis_df(distr_df, df)
print(f"{len(topic_cols)} topics : {topic_cols}")

In [ ]:
yearly = analysis_df.groupby("year")[topic_cols].mean()
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(yearly.T, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(yearly.index))); ax.set_xticklabels(yearly.index)
ax.set_yticks(range(len(topic_cols))); ax.set_yticklabels(topic_cols, fontsize=8)
ax.set_title("Distribution moyenne des topics par année")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()

In [ ]:
gender_df = analysis_df[analysis_df["titulaire-sexe"] != "non mentionné"]
by_gender = gender_df.groupby("titulaire-sexe")[topic_cols].mean()
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(by_gender, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(topic_cols))); ax.set_xticklabels(topic_cols, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(by_gender))); ax.set_yticklabels(by_gender.index)
ax.set_title("Distribution des topics par sexe")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()

In [ ]:
AGE_ORDER = ["entre 20 et 29 ans", "entre 30 et 39 ans", "entre 40 et 49 ans",
             "entre 50 et 59 ans", "entre 60 et 69 ans", "70 ans et plus"]
age_df = analysis_df[analysis_df["titulaire-age-tranche"].isin(AGE_ORDER)]
by_age = age_df.groupby("titulaire-age-tranche")[topic_cols].mean().reindex(AGE_ORDER)
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(by_age, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(topic_cols))); ax.set_xticklabels(topic_cols, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(AGE_ORDER))); ax.set_yticklabels(AGE_ORDER, fontsize=8)
ax.set_title("Distribution des topics par tranche d'âge")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()

In [ ]:
def categorize_mandat(m):
    if "député sortant" in str(m): return "Député sortant"
    if "non mentionné" in str(m): return "Nouveau candidat"
    return "Autre mandat"

analysis_df["statut"] = analysis_df["titulaire-mandat-passe"].apply(categorize_mandat)
by_statut = analysis_df.groupby("statut")[topic_cols].mean()

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(by_statut, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(topic_cols))); ax.set_xticklabels(topic_cols, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(by_statut))); ax.set_yticklabels(by_statut.index)
ax.set_title("Distribution des topics : sortants vs nouveaux")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()

In [ ]:
N = len(topic_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for (label, row), color in zip(by_statut.iterrows(), ["steelblue", "tomato", "green"]):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(topic_cols, fontsize=7)
ax.set_title("Profil thématique : sortants vs nouveaux", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()

In [ ]:
by_dept = analysis_df.groupby("departement-nom")[topic_cols].mean()
by_dept["topic_dominant"] = by_dept.idxmax(axis=1)
by_dept["score_dominant"] = by_dept[topic_cols].max(axis=1)
by_dept = by_dept.reset_index()

geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
with urllib.request.urlopen(geojson_url) as r:
    geojson = json.load(r)

fig = px.choropleth(
    by_dept, geojson=geojson,
    locations="departement-nom", featureidkey="properties.nom",
    color="topic_dominant", title="Topic dominant par département",
    hover_data=["score_dominant"]
)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()

In [ ]:
yearly = analysis_df.groupby("year")[topic_cols].mean()
fig, ax = plt.subplots(figsize=(12, 5))
for col in topic_cols:
    ax.plot(yearly.index, yearly[col], marker="o", label=col)
ax.set_title("Évolution des topics par année")
ax.set_xlabel("Année"); ax.set_ylabel("Poids moyen")
ax.legend(fontsize=7, bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()